# 06. Ruido y filtros de suavizado

**Objetivo:** comprender la convolución, el tratamiento de bordes y cómo promedio, Gaussiano y mediana responden a distintos tipos de ruido.

In [ ]:
import numpy as np
import cv2

from filtrado_digital.io import cargar_imagen, a_grises, ruta_imagen_ejemplo
from filtrado_digital.visualizacion import comparar, mostrar_imagen
from filtrado_digital.sinteticas import agregar_ruido_gaussiano, agregar_ruido_sal_pimienta
from filtrado_digital.filtros import (
    convolucion_2d_manual,
    filtro_promedio_manual,
    filtro_gaussiano_manual,
    filtro_mediana_manual,
    filtro_promedio_opencv,
    filtro_gaussiano_opencv,
    filtro_mediana_opencv,
    kernel_gaussiano,
)

## 1. Ruido, vecindad y kernel

El **ruido** son variaciones no deseadas en los píxeles. Usaremos ruido Gaussiano y ruido sal-pimienta.

La **vecindad** es el conjunto de píxeles cercanos que participa en el cálculo de un nuevo valor. Un **kernel** es una pequeña matriz de pesos que describe cómo participa cada posición de esa vecindad.

Filtrar implica un compromiso: queremos reducir ruido sin eliminar demasiado **detalle útil** ni difuminar los **bordes**, que son cambios importantes de intensidad.

In [ ]:
foto = cargar_imagen(ruta_imagen_ejemplo())
gris = a_grises(foto)
base = gris[220:412, 500:692]

gauss = agregar_ruido_gaussiano(base, sigma=22)
sal_pimienta = agregar_ruido_sal_pimienta(base, proporcion=0.08)
comparar([base, gauss, sal_pimienta], ["Original", "Ruido Gaussiano", "Sal y pimienta"])

## 2. ¿Qué es una convolución?

En un filtro lineal, el kernel se desplaza sobre la imagen. En cada posición:

1. se toma una pequeña ventana de la imagen;
2. se multiplica elemento a elemento por el kernel;
3. se suman los productos;
4. esa suma se convierte en el nuevo valor del píxel.

Veamos una sola posición con una matriz pequeña.

In [ ]:
mini = np.array([
    [10, 10, 10, 10, 10],
    [10, 20, 20, 20, 10],
    [10, 20, 90, 20, 10],
    [10, 20, 20, 20, 10],
    [10, 10, 10, 10, 10],
], dtype=np.uint8)

kernel_promedio = np.ones((3, 3), dtype=np.float64) / 9
ventana_central = mini[1:4, 1:4]
productos = ventana_central * kernel_promedio
nuevo_centro = productos.sum()

print("Ventana central:\n", ventana_central)
print("\nKernel:\n", np.round(kernel_promedio, 3))
print("\nProductos:\n", np.round(productos, 2))
print("\nNuevo valor central:", nuevo_centro)

## 3. ¿Qué ocurre en los bordes? *Padding*

Cuando el kernel llega a una esquina, parte de su vecindad cae fuera de la imagen. Para poder calcular esos píxeles se agrega un borde artificial llamado **padding**.

Tres estrategias comunes son:

- **zero:** agrega ceros alrededor;
- **replicate:** repite el valor del píxel del borde;
- **reflect:** refleja los valores cercanos al borde.

El tratamiento elegido puede cambiar el resultado cerca de los límites de la imagen.

In [ ]:
recorte_borde = base[:64, :64]
resultados_borde = [
    filtro_promedio_manual(recorte_borde, 5, modo_borde="zero"),
    filtro_promedio_manual(recorte_borde, 5, modo_borde="replicate"),
    filtro_promedio_manual(recorte_borde, 5, modo_borde="reflect"),
]
comparar(resultados_borde, ["Zero padding", "Replicate", "Reflect"])

## 4. Filtro promedio y tamaño del kernel

El filtro promedio asigna el mismo peso a todos los píxeles de la vecindad. Un kernel mayor considera una región más amplia: suele suavizar más, pero también puede borrar más detalles.

In [ ]:
comparar(
    [
        gauss,
        filtro_promedio_opencv(gauss, 3),
        filtro_promedio_opencv(gauss, 7),
    ],
    ["Con ruido", "Promedio 3×3", "Promedio 7×7"],
)

## 5. Filtro Gaussiano y `sigma`

El filtro Gaussiano da mayor peso a los píxeles cercanos al centro. `sigma` controla qué tan extendidos están esos pesos:

- un `sigma` pequeño concentra más peso cerca del centro;
- un `sigma` mayor distribuye el peso sobre una región más amplia y produce un suavizado más extendido.

In [ ]:
for sigma in [0.8, 2.0]:
    kernel = kernel_gaussiano(5, sigma)
    print(f"sigma={sigma}\n", np.round(kernel, 3), "\n")

comparar(
    [
        gauss,
        filtro_gaussiano_opencv(gauss, 5, 0.8),
        filtro_gaussiano_opencv(gauss, 5, 2.0),
    ],
    ["Con ruido", "Gauss σ=0.8", "Gauss σ=2.0"],
)

## 6. Filtro de mediana

La mediana no realiza una suma ponderada: ordena los valores de la vecindad y selecciona el central. Por eso suele ser especialmente útil frente al ruido impulsivo sal-pimienta.

In [ ]:
comparar(
    [
        sal_pimienta,
        filtro_promedio_opencv(sal_pimienta, 5),
        filtro_gaussiano_opencv(sal_pimienta, 5, 1.2),
        filtro_mediana_opencv(sal_pimienta, 5),
    ],
    ["Sal y pimienta", "Promedio", "Gaussiano", "Mediana"],
)

## 7. Implementación manual vs OpenCV

Las versiones manuales muestran el algoritmo, mientras que OpenCV utiliza implementaciones optimizadas para aplicaciones reales.

In [ ]:
manual = filtro_gaussiano_manual(gauss, 5, 1.2)
opencv = filtro_gaussiano_opencv(gauss, 5, 1.2)
comparar([manual, opencv], ["Gaussiano manual", "Gaussiano OpenCV"])

## Conclusiones

- La convolución combina una vecindad con un kernel para calcular nuevos píxeles.
- El *padding* define cómo se tratan los límites de la imagen.
- Aumentar el kernel suele aumentar el suavizado y la pérdida de detalle.
- `sigma` controla la extensión de los pesos Gaussianos.
- El filtro adecuado depende del tipo de ruido y de cuánto detalle queremos preservar.